# Lab 11 — Modular Routing with APIRouter + include_router

**Difficulty: Intermediate | ~35 min | Requires Lab 4**

### Step 1: Install Dependencies

This cell installs every pinned dependency the lab needs. Run this first so all later cells have what they require.

In [12]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Imports and App Setup

We import the standard modules, read the OpenRouter API key from the `.env` file, and create the FastAPI application instance.

In [13]:
from fastapi import FastAPI, Depends, APIRouter, HTTPException
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import AsyncOpenAI
import os

load_dotenv()
api_key = os.getenv("OPEN_ROUTER_KEY") or input("Open Router API key: ")
app = FastAPI()

### Step 3: Shared LLM Client and Reply Dependency

We build one `AsyncOpenAI` client at module level and wrap it in a small dependency function. This dependency is used by the endpoint — it exists so the endpoint receives a ready-to-use client without creating one itself.

In [14]:
client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

def get_reply_client():
    return client

### Step 4: Request Model and Router-Level Dependency

We define a Pydantic model for the request body — every POST to the endpoint must include a `message` string.

We also define a **construction-time dependency** and attach it to the router via `APIRouter(dependencies=[...])`. Unlike the mount-time `moderation_gate` we will add later, this dependency is baked into the router itself and will apply to **every endpoint** registered on this router, no matter where or how the router is mounted.

In [15]:
class MessageIn(BaseModel):
    message: str

def api_router_level_dep():
    return "This is a apirouter level dependency applied to every end point"

router = APIRouter(dependencies=[Depends(api_router_level_dep)])

### Step 5: The Message Endpoint

This endpoint contains **no moderation or safety-checking logic whatsoever** — that is the entire point being proven later. It receives the validated `MessageIn` payload, calls the LLM via the injected reply client, and returns the reply. Because it is attached to the `router` object rather than directly to `app`, it can be mounted under different prefixes with different dependencies — the enforcement lives entirely in how and where the router is mounted, not inside this function.

In [16]:
@router.post("/message")
async def send_message(payload: MessageIn, replier=Depends(get_reply_client)):
    reply = await replier.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": payload.message}]
    )
    return {"status": "ok", "reply": reply.choices[0].message.content}

### Step 6: The Router-Level Dependency Endpoint

This endpoint simply returns the value injected by `api_router_level_dep` — proving that the construction-time dependency runs on every request to this endpoint, regardless of which mount prefix was hit. It exists purely to demonstrate that construction-time dependencies apply to all endpoints on the router.

It is declared as a **POST** (rather than a GET) so that the mount-level `MessageIn` body dependency is satisfied on the `/public` mount — more on that interaction in Step 8.

In [17]:
@router.post("/apirouter_dependency")
async def show_router_dep(dep_msg=Depends(api_router_level_dep)):
    return {"message": dep_msg}

### Step 7: The Moderation Gate Dependency

This dependency accepts the same `MessageIn` body model as the endpoint — when both the dependency and the endpoint declare the same Pydantic parameter, FastAPI parses the request body once and both consumers see the same parsed data. If the message text contains the word `unsafe` (case-insensitive), the dependency raises a 400 `HTTPException`. When a dependency raises, FastAPI stops the request immediately and never enters the endpoint function — the blocking happens entirely before the endpoint runs.

In [18]:
def moderation_gate(payload: MessageIn):
    if "unsafe" in payload.message.lower():
        raise HTTPException(status_code=400, detail="blocked by moderation")

### Step 8: Mount the Same Router Twice with Different Dependencies

Here is the core of the lab. The same `router` object — the same function, defined once — is mounted **twice** on the app under different prefixes. The `dependencies=` parameter on `include_router()` is applied **at mount time**: it is not baked into the router itself when it is constructed. This is why the same router can behave differently under different mounts.

- `/public/message` gets `moderation_gate` — the dependency runs before the endpoint on every request.
- `/internal/message` gets no extra dependencies — the endpoint runs directly.

Both mounts still receive the router-level `api_router_level_dep` because it is set at construction time and follows the router everywhere.

This reflects a real pattern in AI systems: public-facing traffic is unpredictable and needs a safety gate, while internal evaluation or testing traffic is already trusted and should not pay that cost.

In [19]:
app.include_router(router, prefix="/public", dependencies=[Depends(moderation_gate)])
app.include_router(router, prefix="/internal")

### Step 9: TestClient Setup

We create a `TestClient` to send simulated HTTP requests against the app directly in the notebook, without running a live server. The following cells prove the two mounts behave differently despite sharing the same endpoint code.

In [20]:
from fastapi.testclient import TestClient

test_client = TestClient(app)

### Proof 1 — Public Mount Blocks Unsafe Input

We POST a message containing the word `unsafe` to the **public** mount. The `moderation_gate` dependency should reject it before the endpoint ever runs, producing a 400 response.

In [21]:
res_public = test_client.post("/public/message", json={"message": "this is unsafe content"})
print(f"Public mount (unsafe message): status={res_public.status_code}")
print(f"Body: {res_public.json()}")

Public mount (unsafe message): status=400
Body: {'detail': 'blocked by moderation'}


### Proof 2 — Internal Mount Passes the Same Payload Freely

We POST the **exact same** message to the **internal** mount. With no moderation dependency, the endpoint processes it normally and returns a 200 with a real LLM reply — the same message that was blocked on the public mount is fully processed here.

In [22]:
res_internal = test_client.post("/internal/message", json={"message": "this is unsafe content"})
print(f"Internal mount (same message): status={res_internal.status_code}")
print(f"Reply: {res_internal.json()['reply'][:120]}...")

Internal mount (same message): status=200
Reply: You're absolutely right to prioritize safety—I'm designed to avoid generating harmful, unsafe, or inappropriate content,...


### Proof 3 — Same Function Object, Not Two Look-Alikes

This is a stronger proof than the two mounts simply looking similar in the code. We walk the app's route table, find the handler function for each mount, and check whether they are the **literal same function object in memory** — proving this is one function serving both paths, not two separate definitions that happen to share a name.

In [23]:
public_handler = None
internal_handler = None

for route in app.routes:
    if getattr(route, "path", None) == "/public/message":
        public_handler = route.endpoint
    if getattr(route, "path", None) == "/internal/message":
        internal_handler = route.endpoint

print(f"Public handler:  {public_handler}")
print(f"Internal handler: {internal_handler}")
print(f"Same function object? {public_handler is internal_handler}")

Public handler:  <function send_message at 0x00000196C9D58540>
Internal handler: <function send_message at 0x00000196C9D58540>
Same function object? True


### Proof 4 — Public Mount Lets Safe Messages Through

The public mount is not a blanket blocker. We send a safe message and confirm it returns 200 with a real generated reply — proving the moderation dependency's own condition is what determines the outcome, not the mount itself.

In [24]:
res_safe = test_client.post("/public/message", json={"message": "What is FastAPI?"})
print(f"Public mount (safe message): status={res_safe.status_code}")
print(f"Reply: {res_safe.json()['reply'][:120]}...")

Public mount (safe message): status=200
Reply: FastAPI is a **high‑performance, modern web framework** for building APIs (and web applications) with Python 3.6+.

### ...


### Proof 5 — Construction-Time Dependency Applies to Both Mounts

The `/apirouter_dependency` endpoint was registered on the same router with `api_router_level_dep` set at construction time. We call it from both the `/public` and `/internal` prefixes to prove the construction-time dependency runs regardless of which mount is hit — it follows the router, not the mount. Each call sends a JSON body with a `message` field, which satisfies the `MessageIn` requirement that the mount-level `moderation_gate` imposes on every `/public` route.

In [25]:
res_pub_dep = test_client.post("/public/apirouter_dependency", json={"message": "any text"})
res_int_dep = test_client.post("/internal/apirouter_dependency", json={"message": "any text"})

print(f"Public mount:  {res_pub_dep.json()}")
print(f"Internal mount: {res_int_dep.json()}")
print(f"Same response? {res_pub_dep.json() == res_int_dep.json()}")

Public mount:  {'message': 'This is a apirouter level dependency applied to every end point'}
Internal mount: {'message': 'This is a apirouter level dependency applied to every end point'}
Same response? True
